In [14]:
ticker = 'RELIANCE'

In [15]:
import polars as pl
import duckdb
from scripts.database import engine


In [16]:
def unified_master_columns(ticker)-> pl.DataFrame:
  query = """
               SELECT "ReportDate", "Volume", "Delivery_Percentage"
FROM unified_market_master
WHERE "InstrumentType" = 'CASH'
order by "ReportDate" ASC
               """
  return engine.execute(
    query
  ).pl()


def unified_matrix_columns(ticker)-> pl.DataFrame:
  query = """
  SELECT 
            date, 
            is_fo_eligible,
            has_block_deal,
            daily_hl_spread, 
            daily_vwap_dev, 
            oi_pcr,
            delta_oi_pcr, 
            futures_basis, 
            net_block_volume,
            avg_block_premium
        FROM mv_unified_market_matrix
        ORDER BY date ASC
  """
  return engine.execute(
    query
  ).pl()

In [ ]:
ticker = 'RELIANCE'

umt = unified_master_columns(ticker)
umc = unified_matrix_columns(ticker)

# Polars Join Syntax
merged_columns = (
    umt.join(umc, on="ReportDate", how="inner")
    .sort("ReportDate", descending=True)
)

print(merged_columns)

AttributeError: 'DuckDBResultContainer' object has no attribute 'pl'

In [ ]:
continuous_features = [
    "Volume", 
    "Delivery_Percentage", 
    "daily_hl_spread", 
    "daily_vwap_dev",
    "oi_pcr", 
    "delta_oi_pcr", 
    "futures_basis", 
    "net_block_volume", 
    "avg_block_premium"
]

# Select the columns, then compute the Pearson correlation matrix natively
correlation_matrix = merged_columns.select(continuous_features).corr()

print(correlation_matrix)

MemoryError: Unable to allocate 54.2 GiB for an array with shape (7278997174,) and data type int64